# The Fourier Transform of the Dirac Comb

This notebook builds intuition for why the Fourier transform of a Dirac comb is itself a Dirac comb.
We start with a single impulse at the origin and progressively add impulses symmetrically, watching
the frequency-domain representation evolve.

For each configuration we show three panels:
- **Real part** of the Fourier transform
- **Imaginary part** of the Fourier transform
- **Power Spectral Density (PSD)** = |F(f)|²

**Reference:** VanderPlas (2018), ApJS 236, 16 — *Understanding the Lomb-Scargle Periodogram*

## Background

The Fourier transform of a time-domain signal $g(t)$ is defined as:

$$\hat{g}(f) = \int_{-\infty}^{\infty} g(t)\, e^{-2\pi i f t}\, dt$$

For a Dirac delta function at position $t_0$, the sifting property gives:

$$\mathcal{F}\{\delta(t - t_0)\} = e^{-2\pi i f t_0} = \cos(2\pi f t_0) - i\sin(2\pi f t_0)$$

Key observations:
- At $t_0 = 0$: transform $= 1$ (purely real, flat spectrum)
- At $t_0 \neq 0$: both real and imaginary parts are non-zero
- The PSD $= |e^{-2\pi i f t_0}|^2 = \cos^2 + \sin^2 = 1$ always — the PSD is flat regardless of impulse location

By linearity, for a sum of impulses:

$$\mathcal{F}\left\{\sum_n \delta(t - t_n)\right\} = \sum_n e^{-2\pi i f t_n}$$

For a **symmetric** arrangement around $t=0$ (i.e. $t_n = \pm nT$), the sine terms cancel and:

$$\hat{g}(f) = 1 + 2\sum_{n=1}^{N} \cos(2\pi f n T) \quad \text{(purely real)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Parameters ────────────────────────────────────────────────────────────────
T   = 1.0                          # impulse spacing (normalised)
N_f = 5000                         # number of frequency points
f   = np.linspace(-4/T, 4/T, N_f)  # frequency axis (show ±4 comb teeth)

# Colour scheme
C_REAL = '#2166ac'   # blue  — real part
C_IMAG = '#d6604d'   # red   — imaginary part
C_PSD  = '#1a9850'   # green — PSD

def plot_transform(f, F, title, ylim_ri=(-3.5, 3.5)):
    """Plot real, imaginary, and PSD panels for a Fourier transform F(f)."""
    PSD = np.abs(F)**2
    fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    fig.suptitle(title, fontsize=13, fontweight='bold')

    axes[0].plot(f, F.real, color=C_REAL, lw=1.5)
    axes[0].axhline(0, color='k', lw=0.5, ls='--')
    axes[0].set_ylabel('Real part', fontsize=10)
    axes[0].set_ylim(ylim_ri)
    axes[0].grid(ls=':', lw=0.5, alpha=0.5)

    axes[1].plot(f, F.imag, color=C_IMAG, lw=1.5)
    axes[1].axhline(0, color='k', lw=0.5, ls='--')
    axes[1].set_ylabel('Imaginary part', fontsize=10)
    axes[1].set_ylim(ylim_ri)
    axes[1].grid(ls=':', lw=0.5, alpha=0.5)

    axes[2].plot(f, PSD, color=C_PSD, lw=1.5)
    axes[2].axhline(0, color='k', lw=0.5, ls='--')
    axes[2].set_ylabel('PSD  |F(f)|²', fontsize=10)
    axes[2].set_xlabel('Frequency  f  (units of 1/T)', fontsize=10)
    axes[2].grid(ls=':', lw=0.5, alpha=0.5)

    # Mark comb tooth positions
    for ax in axes:
        for k in range(-4, 5):
            ax.axvline(k/T, color='grey', lw=0.5, ls=':', alpha=0.4)

    axes[2].set_xlim(f[0], f[-1])
    fig.tight_layout()
    plt.show()

print('Setup complete. T =', T)

---
## Section 1: Single impulse at the origin  $\delta(t=0)$

$$\mathcal{F}\{\delta(t)\} = e^{-2\pi i f \cdot 0} = 1$$

The transform is **1** at every frequency — flat real part, zero imaginary part.
The PSD is also flat at 1.

An impulse at the origin has a perfectly symmetric relationship with the frequency axis:
there is no preferred direction in time, so there is no phase — the imaginary part is identically zero.

In [ ]:
F1 = np.ones_like(f, dtype=complex)   # e^{-2pi i f * 0} = 1
plot_transform(f, F1,
    title=r'Section 1: Single impulse at $t = 0$',
    ylim_ri=(-1.5, 1.5))

---
## Section 2: Single impulse at $t = +T$

$$\mathcal{F}\{\delta(t - T)\} = e^{-2\pi i f T} = \cos(2\pi f T) - i\sin(2\pi f T)$$

Moving the impulse off the origin **activates the imaginary part**.
The phase winds linearly with frequency at a rate proportional to $T$.

Note that the **PSD is still flat at 1** — the timing information is carried entirely
in the phase (imaginary part), which the PSD discards.

In [ ]:
F2 = np.exp(-2j * np.pi * f * T)   # delta at +T
plot_transform(f, F2,
    title=r'Section 2: Single impulse at $t = +T$',
    ylim_ri=(-1.5, 1.5))

---
## Section 3: Single impulse at $t = -T$

$$\mathcal{F}\{\delta(t + T)\} = e^{+2\pi i f T} = \cos(2\pi f T) + i\sin(2\pi f T)$$

The real part is **identical** to Section 2 — cosine is an even function so $\cos(2\pi f T) = \cos(-2\pi f T)$.

But the imaginary part has **flipped sign** — the sine term changes sign with the time shift direction.

The PSD is still flat at 1 — you cannot distinguish $\delta(t-T)$ from $\delta(t+T)$ from the PSD alone.

In [ ]:
F3 = np.exp(+2j * np.pi * f * T)   # delta at -T
plot_transform(f, F3,
    title=r'Section 3: Single impulse at $t = -T$',
    ylim_ri=(-1.5, 1.5))

---
## Section 4: Three impulses — $\delta(-T) + \delta(0) + \delta(+T)$

By linearity:

$$\mathcal{F}\{\delta(t+T) + \delta(t) + \delta(t-T)\} = e^{+2\pi ifT} + 1 + e^{-2\pi ifT} = 1 + 2\cos(2\pi fT)$$

The sine terms from $\pm T$ are **equal and opposite and cancel exactly**.
The imaginary part is identically zero.

This is not a coincidence — it is a theorem:
> **An even function (symmetric about $t=0$) always has a purely real Fourier transform.**

The real part now shows oscillation — the first hint of comb structure emerging.
The PSD sharpens: peaks begin to appear at $f = 0, \pm 1/T$.

In [ ]:
F4 = np.ones_like(f, dtype=complex)          # delta at 0
F4 += np.exp(-2j * np.pi * f * T)            # delta at +T
F4 += np.exp(+2j * np.pi * f * T)            # delta at -T
# Equivalently: 1 + 2*cos(2*pi*f*T)
plot_transform(f, F4,
    title=r'Section 4: Three impulses — $\delta(-T) + \delta(0) + \delta(+T)$')

---
## Section 5: Add pair at $\pm 2T$

$$\hat{g}(f) = 1 + 2\cos(2\pi fT) + 2\cos(4\pi fT)$$

Each symmetric pair contributes another cosine harmonic.
At frequencies $f = k/T$ all cosines equal 1 and add constructively.
Between those frequencies the cosines increasingly cancel.

Imaginary part remains exactly zero — the even-function theorem continues to hold.

In [ ]:
F5 = F4.copy()
F5 += np.exp(-2j * np.pi * f * 2*T)   # delta at +2T
F5 += np.exp(+2j * np.pi * f * 2*T)   # delta at -2T
plot_transform(f, F5,
    title=r'Section 5: Five impulses — pairs at $0, \pm T, \pm 2T$')

---
## Section 6: Add pair at $\pm 3T$

$$\hat{g}(f) = 1 + 2\cos(2\pi fT) + 2\cos(4\pi fT) + 2\cos(6\pi fT)$$

The peaks at $f = k/T$ are sharpening further.
The between-peak regions are increasingly suppressed by destructive interference.

In [ ]:
F6 = F5.copy()
F6 += np.exp(-2j * np.pi * f * 3*T)
F6 += np.exp(+2j * np.pi * f * 3*T)
plot_transform(f, F6,
    title=r'Section 6: Seven impulses — pairs at $0, \pm T, \pm 2T, \pm 3T$')

---
## Section 7: Add pair at $\pm 4T$

$$\hat{g}(f) = 1 + 2\sum_{n=1}^{4}\cos(2\pi f n T)$$

The comb structure is now clearly visible. The peaks are narrow and well-separated,
and the sidelobes between them are small.

In [ ]:
F7 = F6.copy()
F7 += np.exp(-2j * np.pi * f * 4*T)
F7 += np.exp(+2j * np.pi * f * 4*T)
plot_transform(f, F7,
    title=r'Section 7: Nine impulses — pairs at $0, \pm T, \ldots, \pm 4T$')

---
## Section 8: Add pair at $\pm 5T$

$$\hat{g}(f) = 1 + 2\sum_{n=1}^{5}\cos(2\pi f n T)$$

Eleven impulses in time. The Dirac comb in frequency is clearly emerging —
sharp peaks at $f = k/T$, near-zero between them.
In the limit $N \to \infty$ the peaks become true delta functions.

In [ ]:
F8 = F7.copy()
F8 += np.exp(-2j * np.pi * f * 5*T)
F8 += np.exp(+2j * np.pi * f * 5*T)
plot_transform(f, F8,
    title=r'Section 8: Eleven impulses — pairs at $0, \pm T, \ldots, \pm 5T$')

---
## Section 9: Summary — Evolution of the real part and PSD

Here we overlay all configurations to show the progressive convergence to a Dirac comb
in the frequency domain.

Key takeaways:
1. **The imaginary part is identically zero at every step** for symmetric configurations — a consequence of the even-function theorem, not a numerical coincidence.
2. **The real part sharpens progressively** into peaks at $f = k/T$ as more impulse pairs are added.
3. **The PSD sharpens** correspondingly — power concentrates at the harmonic frequencies.
4. **In the limit** of an infinite symmetric train of impulses, the real part converges to a Dirac comb in frequency with spacing $1/T$.
5. **The PSD of a single impulse is always flat at 1**, regardless of its location in time — timing information lives in the phase alone.

In [ ]:
configs = [
    (F1, r'$\delta(0)$',               '#aaaaaa'),
    (F4, r'$\pm T$ added',             '#abd9e9'),
    (F5, r'$\pm 2T$ added',            '#74add1'),
    (F6, r'$\pm 3T$ added',            '#4575b4'),
    (F7, r'$\pm 4T$ added',            '#313695'),
    (F8, r'$\pm 5T$ added (11 total)', '#1a1a6c'),
]

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
fig.suptitle('Summary: Convergence to the Dirac Comb in Frequency', fontsize=13, fontweight='bold')

for F, label, color in configs:
    axes[0].plot(f, F.real,      color=color, lw=1.2, label=label)
    axes[1].plot(f, np.abs(F)**2, color=color, lw=1.2, label=label)

for ax in axes:
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.grid(ls=':', lw=0.5, alpha=0.5)
    for k in range(-4, 5):
        ax.axvline(k/T, color='grey', lw=0.5, ls=':', alpha=0.4)
    ax.legend(fontsize=8, loc='upper right', ncol=2)

axes[0].set_ylabel('Real part of F(f)', fontsize=10)
axes[1].set_ylabel('PSD  |F(f)|²', fontsize=10)
axes[1].set_xlabel('Frequency  f  (units of 1/T)', fontsize=10)
axes[0].set_xlim(f[0], f[-1])

# Annotate comb tooth positions
for k in range(-3, 4):
    axes[1].annotate(f'$f={k}/T$', xy=(k/T, 0), xytext=(k/T, -3),
                     fontsize=7, ha='center', color='grey')

fig.tight_layout()
plt.show()

print('Note: imaginary parts for all symmetric configurations are identically zero')
print('Max |imag| across all symmetric configs:')
for F, label, _ in configs[1:]:
    print(f'  {label:25s}  max|imag| = {np.max(np.abs(F.imag)):.2e}')

---
## Conclusions

This notebook has demonstrated:

1. **A single $\delta(t)$ at the origin** has a flat real spectrum and zero imaginary spectrum.
   PSD = 1 everywhere.

2. **A single $\delta(t \pm T)$ off the origin** has both real (cosine) and imaginary (sine) parts.
   PSD is still flat — the location information is encoded entirely in the phase.

3. **Symmetric pairs** around the origin have zero imaginary parts — a consequence of the
   even-function theorem, not a numerical accident.

4. **Progressive addition** of symmetric pairs causes constructive interference at $f = k/T$
   and destructive interference elsewhere — the Dirac comb in frequency emerges naturally.

5. **The PSD cannot distinguish** the location of a single impulse — that information
   is in the phase, which the PSD discards. This is the fundamental limitation of
   spectrum analyzers and periodogram methods when phase information is needed.

This result is the mathematical foundation of the Nyquist-Shannon sampling theorem
and is central to understanding aliasing in the Lomb-Scargle periodogram
(VanderPlas 2018, Section 3).